# Simulate Cityfow

### Change directory to cityflow

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

# Tìm thư mục gốc của project
current_dir = Path.cwd()
root_dir = current_dir

while root_dir != root_dir.parent:
  if (root_dir / "requirements.txt").exists() or (root_dir / ".git").exists():
    break
  root_dir = root_dir.parent

# Xác định thư mục CityFlow
cityflow_dir = root_dir / "cityflow"

# Chuyển working directory vào CityFlow
os.chdir(cityflow_dir)

# Thêm root và cityflow vào sys.path
for path in [root_dir, cityflow_dir]:
  if str(path) not in sys.path:
    sys.path.insert(0, str(path))

print("Project root:", root_dir)
print("Working directory:", Path.cwd())

#### Settup

In [ ]:
import sys
from pathlib import Path
import json
import cityflow

MAIN_DIR = Path("sample/test/test_1/")

## 1. Generate Roadnet

In [ ]:
from src.generators.roadnet_gen import generate_roadnet

roadnet_path = generate_roadnet(
  rows=2,
  cols=3,
  output_dir=MAIN_DIR,
  roadnet_name="roadnet.json",
  num_left_lanes=1,
  num_straight_lanes=2,
  num_right_lanes=1,
  row_distance=300,
  column_distance=300,
  tl_plan=True,   # thêm --tlPlan (kế hoạch đèn tín hiệu)
  turn=True,      # thêm --turn (cho phép rẽ)
)

with open(roadnet_path, encoding="utf-8") as f:
  net = json.load(f)

real = [i for i in net["intersections"] if not i["virtual"]]
print("Ngã tư thật:", len(real))
print("Số road:", len(net["roads"]))

## 2. Generate Flow (Nên dùng fiel temp_flow.json đã tạo ở #1, mục này dag thử nghiệm)

In [ ]:
from src.generators.flow_gen import FlowGenerator
from src.vehicles import VehicleFleet


gen_flow = FlowGenerator(
  roadnet_json_file=MAIN_DIR / "roadnet.json",
  flow_json_file=MAIN_DIR / "flow.json"
)
gen_flow.generate(
  interval=10,     
  startTime=0,
  endTime=-1,        
  vehicles=[(0.8, VehicleFleet.CAR), (0.2, VehicleFleet.TRUCK)],
  seed=42,       
)
print("Virtual nodes:", len(gen_flow.virtual_nodes))
print("Số flow:", len(gen_flow.flows))
print("Đã lưu:", gen_flow.flow_json_file)

## SIMULATE

In [ ]:
import cityflow

from src.generators.config_gen import gen_config
config = gen_config(
  dir=MAIN_DIR,
  config_name="config.json",
  roadnet_name="roadnet.json",
  flow_name="temp_flow.json",
  rl_traffic_light=False,
)

TOTAL_STEPS = 1000
LOG_INTERVAL = 10

engine = cityflow.Engine(str(MAIN_DIR / "config.json"), thread_num=1)
try:
  for step in range(1, TOTAL_STEPS + 1):
    engine.next_step()
    if step % LOG_INTERVAL == 0 or step == TOTAL_STEPS:
      print(f"[Step {step:04d}/{TOTAL_STEPS}] Vehicles: {engine.get_vehicle_count()}")

  print("Xe còn lại:", engine.get_vehicle_count())
  print("Avg travel time:", engine.get_average_travel_time())
finally:
  del engine